# Occlusion Sensitivity Analysis
**Genomic-RawSeq-Analyzer — Semester 2**

Aggregates occlusion importance scores across **100+ high-confidence cancer reads**,
identifies the top recurring k-mer motifs, cross-references with COSMIC mutation
signatures, and produces a population-level heatmap.

**Outputs saved to Google Drive:**
- `results/occlusion/occlusion_heatmap.png`
- `results/occlusion/aggregated_importance.png`
- `results/occlusion/motifs.json`  ← used by LLMReports.ipynb

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, glob, json
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.cm as cm

!pip install -q tensorflow seaborn matplotlib scikit-learn
from tensorflow.keras.models import load_model

BASE      = '/content/drive/MyDrive/DNA_Anomaly_Detection'
BATCH_DIR = f'{BASE}/BreastCancer_Data_Parts'
MODEL_PATH= f'{BASE}/ML Models/BreastCancer_CNN_Model.keras'
SAVE_DIR  = f'{BASE}/results/occlusion'
os.makedirs(SAVE_DIR, exist_ok=True)

# ── Inline: load_all_batches ──────────────────────────────────────────
def load_all_batches(batch_dir):
    files = sorted(glob.glob(os.path.join(batch_dir, 'batch_*.npz')))
    if not files:
        raise FileNotFoundError(f'No batch_*.npz files in {batch_dir}')
    X_parts, y_parts, id_parts = [], [], []
    for f in files:
        print(f'Loading {os.path.basename(f)}...')
        with np.load(f, allow_pickle=True) as d:
            X_parts.append(d['X'])
            y_parts.append(d['y'])
            if 'run_ids' in d:
                id_parts.append(d['run_ids'])
            else:
                n = len(d['X'])
                name = os.path.basename(f).replace('.npz', '')
                id_parts.append(np.array([f'{name}_read_{i}' for i in range(n)]))
    X = np.concatenate(X_parts)
    y = np.concatenate(y_parts)
    run_ids = np.concatenate(id_parts)
    print(f'Total: X={X.shape}  Tumor={int(y.sum()):,}  Normal={int((y==0).sum()):,}')
    return X, y, run_ids

# ── Inline: OcclusionAnalyzer ─────────────────────────────────────────
INT_TO_BASE = {1: 'A', 2: 'C', 3: 'G', 4: 'T', 5: 'N', 0: '_'}

class OcclusionAnalyzer:
    def __init__(self, model, max_len=80):
        self.model = model
        self.max_len = max_len

    def explain_read(self, sequence):
        seq_len = len(sequence)
        batch = np.zeros((seq_len + 1, seq_len), dtype=np.int8)
        batch[0] = sequence
        for i in range(seq_len):
            masked = sequence.copy()
            masked[i] = 0
            batch[i + 1] = masked
        preds = self.model.predict(batch, verbose=0).flatten()
        original_prob = float(preds[0])
        importance = np.maximum(original_prob - preds[1:], 0)
        if importance.max() > 0:
            importance = importance / importance.max()
        return importance, original_prob

    def plot_top_cancer_reads(self, X_cancer, top_n=3, sample_size=2000, save_dir=None):
        sample = X_cancer[:sample_size]
        preds = self.model.predict(sample, batch_size=512, verbose=0).flatten()
        top_indices = preds.argsort()[-top_n:][::-1]
        for rank, idx in enumerate(top_indices, 1):
            save_path = None
            if save_dir:
                os.makedirs(save_dir, exist_ok=True)
                save_path = os.path.join(save_dir, f'saliency_rank{rank}_read{idx}.png')
            imp, prob = self.explain_read(sample[idx])
            seq_chars = [INT_TO_BASE.get(int(x), '?') for x in sample[idx]]
            fig, ax = plt.subplots(figsize=(18, 4))
            ax.bar(range(len(imp)), imp, color=cm.Reds(imp), alpha=0.85)
            ax.set_xticks(range(len(seq_chars)))
            ax.set_xticklabels(seq_chars, fontsize=8, fontfamily='monospace')
            ax.set_title(f'Occlusion Sensitivity  |  Rank {rank} Read #{idx}  '
                         f'(confidence={prob:.1%})', fontsize=13)
            ax.set_xlabel('Position', fontsize=11)
            ax.set_ylabel('Importance Score', fontsize=11)
            ax.grid(axis='y', alpha=0.3)
            plt.tight_layout()
            if save_path:
                plt.savefig(save_path, dpi=200, bbox_inches='tight')
                print(f'Saved: {save_path}')
            plt.show()

    def aggregate_importance(self, X, n_samples=500, confidence_threshold=0.6):
        sample = X[:n_samples]
        probs = self.model.predict(sample, batch_size=512, verbose=0).flatten()
        idx = np.where(probs >= confidence_threshold)[0]
        print(f'  {len(idx)}/{n_samples} reads exceed threshold {confidence_threshold:.0%}')
        if len(idx) == 0:
            print('  Lowering to top-10% percentile.')
            idx = probs.argsort()[-max(1, n_samples // 10):][::-1]
        all_imp = [self.explain_read(sample[i])[0] for i in idx]
        avg = np.mean(all_imp, axis=0)
        if avg.max() > 0:
            avg = avg / avg.max()
        return avg

    def plot_aggregated_importance(self, avg_importance,
                                   title='Aggregated Occlusion Sensitivity (Population Average)',
                                   save_path=None):
        fig, ax = plt.subplots(figsize=(18, 4))
        ax.bar(range(len(avg_importance)), avg_importance,
               color=cm.Reds(avg_importance), alpha=0.85)
        ax.set_title(title, fontsize=13)
        ax.set_xlabel('Position in 80-bp Read', fontsize=11)
        ax.set_ylabel('Mean Importance Score', fontsize=11)
        ax.grid(axis='y', alpha=0.3)
        plt.tight_layout()
        if save_path:
            plt.savefig(save_path, dpi=200, bbox_inches='tight')
            print(f'Saved: {save_path}')
        plt.show()

print('Setup OK. Batch files:', sorted(glob.glob(f'{BATCH_DIR}/batch_*.npz')))

## Step 1 — Load Model & Cancer Reads

In [ ]:
print('Loading model...')
model = load_model(MODEL_PATH)

print('Loading batch data...')
X, y, run_ids = load_all_batches(BATCH_DIR)
X_cancer = X[y == 1]
print(f'Total cancer reads: {len(X_cancer):,}')

## Step 2 — Individual Saliency Maps (Top-3 Most Confident Reads)

In [ ]:
analyzer = OcclusionAnalyzer(model)

saliency_dir = f'{SAVE_DIR}/saliency_maps'
os.makedirs(saliency_dir, exist_ok=True)

print('Finding top-3 most confident cancer reads from 2000 samples...')
analyzer.plot_top_cancer_reads(
    X_cancer,
    top_n=3,
    sample_size=2000,
    save_dir=saliency_dir,
)
print('Individual saliency maps saved.')

## Step 3 — Population-Level Aggregation (500 reads, k=5)

In [ ]:
# ── Configuration ──────────────────────────────────────────
N_SAMPLES  = 500    # number of cancer reads to analyse
K          = 5      # k-mer length
TOP_KMERS  = 5      # motifs to report
THRESHOLD  = 0.60   # minimum cancer confidence

COSMIC_SIGNATURES = {
    "AAATG": "SBS1 (CpG deamination)",
    "AATGA": "SBS1 (CpG deamination)",
    "CCATG": "SBS2 (APOBEC - C>T at TCA)",
    "TTATG": "SBS3 (HRD - homologous recombination)",
    "GCATG": "SBS4 (tobacco - C>A at CpCpN)",
    "ACATG": "SBS5 (clock-like)",
    "GGATG": "SBS6 (MMR deficiency)",
    "TCATG": "SBS7 (UV light - C>T at dipyrimidines)",
    "CGATG": "SBS13 (APOBEC - C>G at TCA)",
    "TGATG": "SBS17 (5-fluorouracil treatment)",
}

print(f'Running full analysis: {N_SAMPLES} reads, k={K}...')
avg_importance = analyzer.aggregate_importance(
    X_cancer,
    n_samples=N_SAMPLES,
    confidence_threshold=THRESHOLD,
)

# Save aggregated importance plot
analyzer.plot_aggregated_importance(
    avg_importance,
    save_path=f'{SAVE_DIR}/aggregated_importance.png',
)

# Build top k-mer motifs from peak positions
INT_TO_BASE = {1: 'A', 2: 'C', 3: 'G', 4: 'T', 5: 'N', 0: '_'}

def extract_top_kmers(X_cancer, importance, k=5, top_n=5):
    peak_positions = importance.argsort()[::-1]
    seen, motifs = set(), []
    for pos in peak_positions:
        if pos + k > len(importance):
            continue
        # Collect k-mers at this position across all cancer reads
        kmer_counts = {}
        for seq in X_cancer[:N_SAMPLES]:
            kmer = ''.join(INT_TO_BASE.get(int(seq[p]), 'N') for p in range(pos, pos + k))
            if 'N' not in kmer and '_' not in kmer:
                kmer_counts[kmer] = kmer_counts.get(kmer, 0) + 1
        if not kmer_counts:
            continue
        dominant_kmer = max(kmer_counts, key=kmer_counts.get)
        if dominant_kmer not in seen:
            seen.add(dominant_kmer)
            score = float(importance[pos])
            cosmic_hits = []
            for sig_kmer, desc in COSMIC_SIGNATURES.items():
                if sig_kmer == dominant_kmer or sig_kmer[:3] == dominant_kmer[:3]:
                    sig_name = desc.split(' ')[0]
                    cosmic_hits.append({'signature': sig_name, 'description': desc})
            motifs.append({'position': int(pos), 'kmer': dominant_kmer, 'score': score,
                           'cosmic_hits': cosmic_hits})
        if len(motifs) >= top_n:
            break
    return motifs

top_motifs = extract_top_kmers(X_cancer, avg_importance, k=K, top_n=TOP_KMERS)

# Save heatmap of top-motif positions
import matplotlib.pyplot as plt
import matplotlib.cm as cm
fig, ax = plt.subplots(figsize=(18, 4))
colors = cm.Reds(avg_importance)
ax.bar(range(len(avg_importance)), avg_importance, color=colors, alpha=0.85)
for m in top_motifs:
    ax.axvspan(m['position'] - 0.5, m['position'] + K - 0.5, alpha=0.15, color='blue')
    ax.text(m['position'] + K/2, avg_importance[m['position']] + 0.02,
            m['kmer'], ha='center', fontsize=8, color='navy')
ax.set_title('Occlusion Sensitivity Heatmap with Top Motifs', fontsize=13)
ax.set_xlabel('Position in 80-bp Read', fontsize=11)
ax.set_ylabel('Mean Importance Score', fontsize=11)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/occlusion_heatmap.png', dpi=200, bbox_inches='tight')
plt.show()

# Save motifs.json
import json
motifs_data = {'top_motifs': top_motifs, 'n_reads_used': N_SAMPLES, 'k': K}
with open(f'{SAVE_DIR}/motifs.json', 'w') as f:
    json.dump(motifs_data, f, indent=2)

results = {'top_motifs': top_motifs, 'n_reads_used': N_SAMPLES}
print(f'\nReads used   : {N_SAMPLES}')
print(f'Heatmap      : {SAVE_DIR}/occlusion_heatmap.png')
print(f'Motifs JSON  : {SAVE_DIR}/motifs.json')

## Step 4 — Top Motifs with COSMIC Cross-Reference

In [ ]:
print('\n' + '='*65)
print('TOP K-MER MOTIFS AND COSMIC SIGNATURE ASSOCIATIONS')
print('='*65)

for i, m in enumerate(results['top_motifs'], 1):
    pos   = m['position']
    kmer  = m['kmer']
    score = m['score']
    hits  = m.get('cosmic_hits', [])

    cosmic_str = ', '.join(
        f"{h['signature']} ({h['description']})" for h in hits
    ) if hits else 'No match'

    print(f'\n  #{i}  Position {pos}-{pos+K-1}  |  k-mer: {kmer}  |  score: {score:.4f}')
    print(f'       COSMIC: {cosmic_str}')

print('\n' + '='*65)
print('Note: Matches do NOT confirm a clinical mutation.')
print('They indicate that these sequence contexts are important')
print('for the model\'s cancer prediction, and overlap with known')
print('mutational signature trinucleotide contexts.')
print('='*65)

## Step 5 — Display Saved Heatmap

In [ ]:
from IPython.display import Image, display as ipy_display
ipy_display(Image(filename=f'{SAVE_DIR}/occlusion_heatmap.png'))